In [2]:
import re
import requests
import pandas as pd

uniprot_pattern = re.compile(
    r"^(?:"
    r"[OPQ][0-9][A-Z0-9]{3}[0-9]"                    # old 6-char pattern 1
    r"|"
    r"[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2}"  # old/new 6-char and 10-char
    r")(?:-\d+)?$"
)
ensembl_pattern = re.compile(
    r"^ENS[A-Z]{0,10}(?:E|FM|G|GT|P|R|T)\d{6,}(?:\.\d+)?$"
)

def get_uniprot(uniprot_id: str):
    '''
    define http_function to get the data from Uniprot API
    '''
    endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
    return requests.get(endpoint, params={"accessions": uniprot_id}, timeout=30)

def uniprot_parse_response(resp):
    '''
    parse response from Uniprot and output
    organism, geneInfo, sequenceInfo, type

    do not forget to include error handling (e.g. for key errors)
    '''
    try:
        payload = resp.json()
        results = payload.get("results", [])
        output = {}
        for val in results:
            acc = val.get("primaryAccession")
            output[acc] = {
                "organism": val.get("organism", {}).get("scientificName"),
                "geneInfo": val.get("genes"),
                "sequenceInfo": val.get("sequence"),
                "type": "protein"
            }
        if not output:
            return {"error": "No records found"}
        return output
    except (ValueError, TypeError, KeyError) as e:
        return {"error": str(e)}

def get_ensembl(ensembl_id: str):
    '''
    define http_function to get the data from Ensembl API
    '''
    endpoint = f"https://rest.ensembl.org/lookup/id/{ensembl_id}"
    return requests.get(endpoint, params={"content-type": "application/json"}, timeout=30)

def ensembl_parse_response(resp):
    '''
    parse Ensembl response and output
    object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

    do not forget to include error handling (e.g. for key errors)
    '''
    try:
        data = resp.json()
        ens_id = data.get("id")
        row = {
            "object_type": data.get("object_type"),
            "species": data.get("species"),
            "assembly_name": data.get("assembly_name"),
            "biotype": data.get("biotype"),
            "display_name": data.get("display_name"),
            "id": data.get("id"),
            "db_type": data.get("db_type"),
            "description": data.get("description"),
            "source": data.get("source"),
            "canonical_transcript": data.get("canonical_transcript")
        }
        if not ens_id:
            return {"error": "No records found"}
        return {ens_id: row}
    except (ValueError, TypeError, KeyError) as e:
        return {"error": str(e)}

def main(ids: list):
    '''
    Function that iterates over all the provided IDs, parses them,
    transforms the results into one pandas.DataFrame, and returns it.

    Invalid IDs and request errors are stored in the error column.
    '''
    rows = []
    for current_id in ids:
        current_id = current_id.strip()
        if not current_id:
            continue
        try:
            if ensembl_pattern.match(current_id):
                resp = get_ensembl(current_id)
                resp.raise_for_status()
                parsed = ensembl_parse_response(resp)
            elif uniprot_pattern.match(current_id):
                resp = get_uniprot(current_id)
                resp.raise_for_status()
                parsed = uniprot_parse_response(resp)
            else:
                rows.append({"input_id": current_id, "error": "error:unknown database"})
                continue

            if "error" in parsed:
                rows.append({"input_id": current_id, "error": parsed["error"]})
            else:
                if current_id in parsed:
                    row = parsed[current_id]
                else:
                    row = next(iter(parsed.values()))
                rows.append({"input_id": current_id, **row})
        except requests.RequestException as e:
            rows.append({"input_id": current_id, "error": str(e)})
    return pd.DataFrame(rows)




In [3]:
with open("test_data.txt") as f:
    test_ids = [line.strip() for line in f if line.strip()]

results = main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618', 'MGP_AJ_G0023128'])

In [4]:
results

,input_id,organism,geneInfo,sequenceInfo,type,error,object_type,species,assembly_name,biotype,display_name,id,db_type,description,source,canonical_transcript
0,P11473,Homo sapiens,[{'geneName': {'evidences': [{'evidenceCode': ...,{'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFH...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q91XI3,Ictidomys tridecemlineatus,[{'geneName': {'value': 'INS'}}],{'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHL...,protein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,hello,NaN,NaN,NaN,NaN,error:unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENSG00000157764,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRAF,ENSG00000157764,core,"B-Raf proto-oncogene, serine/threonine kinase ...",ensembl_havana,ENST00000646891.2
4,ENSG00000139618,NaN,NaN,NaN,NaN,NaN,Gene,homo_sapiens,GRCh38,protein_coding,BRCA2,ENSG00000139618,core,BRCA2 DNA repair associated [Source:HGNC Symbo...,ensembl_havana,ENST00000380152.8
5,MGP_AJ_G0023128,NaN,NaN,NaN,NaN,error:unknown database,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# # Examples:

# get_uniprot('P11473')
# >>> <Response [200]>

# get_uniprot('helloworld')
# >>> <Response [400]>

# get_uniprot('helloworld').json()
# >>> {'url': 'http://rest.uniprot.org/uniprotkb/accessions',
#  'messages': ["Accession 'helloworld' has invalid format. It should be a valid UniProtKB accession with optional sequence range e.g. P12345[10-20]."]}

# uniprot_parse_response(get_uniprot('P11473'))
# >>>
# {'P11473': {'organism': 'Homo sapiens',
#   'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
#        'source': 'HGNC',
#        'id': 'HGNC:12679'}],
#      'value': 'VDR'},
#     'synonyms': [{'value': 'NR1I1'}]}],
#   'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
#    'length': 427,
#    'molWeight': 48289,
#    'crc64': 'F95F300D042C4CB7',
#    'md5': '0D963ACD4A34674368324EE026023597'},
#   'type': 'protein'}
# }

# get_ensembl('ENSMUSG00000041147')
# >>> <Response [200]>

# get_ensembl('helloworld')
# >>> <Response [400]>

# get_ensembl('helloworld').json()
# >>> {'error': "ID 'helloworld' not found"}
# ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))


# >>>
# { 'ENSMUSG00000041147': {'object_type': 'Gene',
#   'species': 'mus_musculus',
#   'assembly_name': 'GRCm39',
#   'biotype': 'protein_coding',
#   'display_name': 'Brca2',
#   'id': 'ENSMUSG00000041147',
#   'db_type': 'core',
#   'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
#   'source': 'ensembl_havana',
#   'canonical_transcript': 'ENSMUST00000044620.11'}
#   }

# main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618', 'MGP_AJ_G0023128'])
# >>> pandas.DataFrame with one row per input ID and an error column for invalid entries
